# An unfitted time-dependant Stokes problem

We can solve the Stokes problem on a moving (unfitted) domain. Here we show an example of a rising ball that the fluid moves around.

In [ ]:
from ngsolve import *
from xfem import *
import ngsolve.webgui as ngw
from netgen.occ import *

from ngsxditto.transport import *
from ngsxditto.levelset import *
from ngsxditto.fluid import *
from ngsxditto.redistancing import *
from ngsxditto.solver import *

In [ ]:
domain = MoveTo(-1, -1).Rectangle(2, 2).Face()
domain.edges.Max(X).name = "right"
domain.edges.Min(X).name = "left"
domain.edges.Min(Y).name = "bottom"
domain.edges.Max(Y).name = "top"
mesh = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=0.1))

In [ ]:
dt = 0.05
t = Parameter(0)
starting_levelset = -((x**2 + (y+0.5-t)**2)**(1/2) - 0.5)
transport = KnownSolutionTransport(mesh, starting_levelset, dt=dt, time=t)
levelset = LevelSetGeometry(transport)
levelset.Initialize(starting_levelset)
ngw.Draw(levelset.field)

We define the fluid and solve the stationary Stokes problem for the starting levelset position.

In [ ]:
fluid_params = FluidParameters(viscosity=1e-1)
fluid = TaylorHood(mesh, fluid_params, lset=levelset, dt=dt, if_dirichlet=CF((0, 0)))
dirichlet = {"left": CF(((1-y)*(1+y), 0)), "top": CF((0, 0)), "bottom": CF((0, 0))}
fluid.Initialize(dirichlet=dirichlet)
sol = fluid.SolveStokes()
fluid.SetInitialValues(sol.components[0], sol.components[1])
ngw.Draw(IfPos(levelset.field, CF((0, 0)), fluid.gfu.components[0]), mesh)
ngw.Draw(IfPos(levelset.field, CF(0), fluid.gfu.components[1]), mesh)

In [ ]:
solver = Solver(fluid=fluid, time=t)
solver.Add(solver.lset.OneStep)
solver.Add(solver.fluid.OneStep)

T = 1
solver.Do(end_time=T)